# Machine Failure Risk & Maintenance Predictor

In [1]:
from pathlib import Path
import time
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, classification_report, confusion_matrix

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "ai4i2020.csv"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_PATH = MODEL_DIR / "machine_failure_model.pkl"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]
TARGET = "Machine failure"

print("Project:", PROJECT_DIR)
print("Dataset:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())
print("Model output:", MODEL_PATH)

Project: D:\HW
Dataset: D:\HW\data\ai4i2020.csv
Dataset exists: True
Model output: D:\HW\models\machine_failure_model.pkl


## 1. Load and validate dataset

If the dataset is not found, place `ai4i2020.csv` inside the project's `data` folder and run the notebook from the project root.

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
required_columns = FEATURES + [TARGET]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing dataset columns: {missing_columns}")

df = df[required_columns].dropna().copy()
df[TARGET] = df[TARGET].astype(int)

print("Dataset shape:", df.shape)
display(df.head())
print("Target counts:")
display(df[TARGET].value_counts().rename(index={0: "No Failure", 1: "Failure"}))

Dataset shape: (10000, 7)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


Target counts:


Machine failure
No Failure    9661
Failure        339
Name: count, dtype: int64

## 2. Choose testing percentage

`TEST_SIZE_PERCENT = 20` means **80% training and 20% testing**. This split is used for evaluation.

In [3]:
TEST_SIZE_PERCENT = 20  # Set an integer from 1 to 99
CV_FOLDS = 5

if not 1 <= TEST_SIZE_PERCENT <= 99:
    raise ValueError("TEST_SIZE_PERCENT must be between 1 and 99.")
if CV_FOLDS < 2:
    raise ValueError("CV_FOLDS must be at least 2.")

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE_PERCENT / 100,
    random_state=42,
    stratify=y
)

print(f"Training rows: {len(X_train)} ({100 - TEST_SIZE_PERCENT}%)")
print(f"Testing rows: {len(X_test)} ({TEST_SIZE_PERCENT}%)")

Training rows: 8000 (80%)
Testing rows: 2000 (20%)


## 3. Create preprocessing and model pipeline

The model excludes identifiers and failure-mode flags to avoid target leakage.

In [4]:
def build_model():
    preprocessor = ColumnTransformer(
        transformers=[
            ("type_encoder", OneHotEncoder(handle_unknown="ignore"), ["Type"])
        ],
        remainder="passthrough"
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

model = build_model()
model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('type_encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Type'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', n_jobs=-1,
                                        random_state=42))])

## 4. Evaluate model

These metrics are for project development and should not be shown on the end-user result page.

In [5]:
minority_count = int(y_train.value_counts().min())
if CV_FOLDS > minority_count:
    raise ValueError(f"CV_FOLDS must be no more than {minority_count} for this training split.")

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
start_time = time.time()

cv_scores = cross_val_score(
    model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=1
)
model.fit(X_train, y_train)
test_predictions = model.predict(X_test)
elapsed_seconds = round(time.time() - start_time, 2)

print(f"Cross-validation accuracy: {cv_scores.mean() * 100:.2f}%")
print(f"Test accuracy: {accuracy_score(y_test, test_predictions) * 100:.2f}%")
print(f"Test precision: {precision_score(y_test, test_predictions, zero_division=0) * 100:.2f}%")
print(f"Training/evaluation time: {elapsed_seconds} seconds")
print("\nClassification report:")
print(classification_report(y_test, test_predictions, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_predictions))

Cross-validation accuracy: 97.75%
Test accuracy: 98.10%
Test precision: 96.88%
Training/evaluation time: 4.58 seconds

Classification report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.97      0.46      0.62        68

    accuracy                           0.98      2000
   macro avg       0.97      0.73      0.81      2000
weighted avg       0.98      0.98      0.98      2000

Confusion matrix:
[[1931    1]
 [  37   31]]


## 5. Train final deployment model and save it

The test metrics above come from a held-out test split. After evaluation, a final model is fit on all available rows so the deployed model can use the full dataset.

In [6]:
final_model = build_model()
final_model.fit(X, y)

joblib.dump(final_model, MODEL_PATH)
print("Model saved successfully:", MODEL_PATH.resolve())
print("File exists:", MODEL_PATH.exists())

Model saved successfully: D:\HW\models\machine_failure_model.pkl
File exists: True


## 6. Verify saved model with example inputs

The model may return the same prediction for different examples. Do not force predictions to change; inspect actual model output.

In [7]:
saved_model = joblib.load(MODEL_PATH)

sample_inputs = pd.DataFrame([
    {
        "Type": "M",
        "Air temperature [K]": 300.0,
        "Process temperature [K]": 310.0,
        "Rotational speed [rpm]": 1500,
        "Torque [Nm]": 35.0,
        "Tool wear [min]": 10
    },
    {
        "Type": "L",
        "Air temperature [K]": 300.0,
        "Process temperature [K]": 310.0,
        "Rotational speed [rpm]": 1300,
        "Torque [Nm]": 60.0,
        "Tool wear [min]": 240
    }
])

predictions = saved_model.predict(sample_inputs)
results = sample_inputs.copy()
results["Prediction"] = [
    "Failure Predicted" if value == 1 else "No Failure Predicted"
    for value in predictions
]

if hasattr(saved_model, "predict_proba"):
    classes = list(saved_model.named_steps["classifier"].classes_)
    failure_column = classes.index(1)
    results["Estimated Failure Probability (%)"] = (
        saved_model.predict_proba(sample_inputs)[:, failure_column] * 100
    ).round(2)

display(results)

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Prediction,Estimated Failure Probability (%)
0,M,300.0,310.0,1500,35.0,10,No Failure Predicted,0.0
1,L,300.0,310.0,1300,60.0,240,Failure Predicted,80.0


## Flask connection

This notebook saves the model to `models/machine_failure_model.pkl`, the same path used by your Flask app. Run all notebook cells successfully, then start Flask and choose **Existing Model** on the model-choice page. Choose the training option in Flask whenever you want to train and save a new model with its selected settings.

Keep the notebook in the project root next to `app.py`. Ensure the notebook and Flask environment use compatible scikit-learn versions.